## INIT

In [64]:
from sources import datasets, to_pd

In [65]:
center = [
    datasets.get('Zoning').geometry.centroid.y.mean(),
    datasets.get('Zoning').geometry.centroid.x.mean()
]

/tmp/ipykernel_244927/1681914242.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  datasets.get('Zoning').geometry.centroid.y.mean(),
/tmp/ipykernel_244927/1681914242.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  datasets.get('Zoning').geometry.centroid.x.mean()


## ENGINEERING

In [66]:
# Create a new column 'day_type' based on trip_id
gdf=datasets.get('routes')

def classify_trip(trip_id: str):
    if "Weekday" in trip_id:
        return "Weekday"
    elif "Saturday" in trip_id:
        return "Saturday"
    elif "Sunday" in trip_id:
        return "Sunday"
    else:
        return "Other"
    
gdf["day_type"] = gdf["trip_id"].apply(classify_trip)

routes_weekday = gdf[gdf["day_type"] == "Weekday"]
routes_saturday = gdf[gdf["day_type"] == "Saturday"]
routes_sunday = gdf[gdf["day_type"] == "Sunday"]
routes_other = gdf[gdf["day_type"] == "Other"]

In [67]:
# Service Exceptions Popup
import pandas as pd
import json

def service_popup(row):

    exceptions = row["service_exceptions"]

    if isinstance(exceptions, str):
        exceptions = json.loads(exceptions)

    if exceptions:
        df = pd.DataFrame(exceptions)
        table = df.to_html(index=False)
    else:
        table = "<p>No service exceptions</p>"

    header = f"""
    <b>Route Short Name:</b> {row['route_short_name']}<br>
    <b>Route ID:</b> {row['route_id']}<br>
    <b>Route Long Name:</b> {row['route_long_name']}<br>
    <b>Trip ID:</b> {row['trip_id']}<br>
    <hr>
    """

    scrollable = f"""
    <div style="max-height:200px; overflow-y:auto;">
        {table}
    </div>
    """

    return header + scrollable

In [68]:
# Stops Popup
import pandas as pd
import json


def stops_popup(row):

    stop_times = row["stop_times"]

    if isinstance(stop_times, str):
        stop_times = json.loads(stop_times)

    if not stop_times:
        return "<b>No stop times available</b>"

    df = pd.DataFrame(stop_times)

    # classify day type
    df["day_type"] = df["trip_id"].apply(classify_trip)

    # sort times
    df = df.sort_values("arrival_time")

    header = f"""
    <b>Stop ID:</b> {row['stop_id']}<br>
    <b>Stop Name:</b> {row['stop_name']}<br>
    <hr>
    """

    html_sections = []

    for day_type, group in df.groupby("day_type"):

        table = group[[
            "trip_id",
            "arrival_time",
            "departure_time",
            "stop_sequence"
        ]].to_html(index=False)

        section = f"""
        <b>{day_type}</b>
        {table}
        <br>
        """

        html_sections.append(section)

    scrollable = f"""
    <div style="max-height:250px; overflow-y:auto;">
        {''.join(html_sections)}
    </div>
    """

    return header + scrollable

In [69]:
import folium
m = folium.Map(
    location=center, 
    zoom_start=13, 
    tiles=None
)
# positron
folium.TileLayer(
    tiles="CartoDB positron",
    name="Positron"
).add_to(m)

# Dark map
folium.TileLayer(
    tiles="CartoDB dark_matter",
    name="Dark"
).add_to(m)

# Light map
folium.TileLayer(
    tiles="OpenStreetMap",
    name="Light"
).add_to(m)

In [70]:
# bus route colors, style
route_colors = {
    '9-BUSINESS PARK': "#D1F30E",
    '3-BROOKDALE': "#FFE600",
    '2-SUNRISE': "#EC9A02",
    '5-MCCONNELL': "#0D7EFF",
    '4-RIVERDALE': "#95FD59",
    '6-CUMBERLAND': "#29FFCA",
    'COMMUNITY SERVICE WEST': "#A8A8A8",
    '8-BUSINESS PARK': "#0299F0",
    'COMMUNITY SERVICE EAST': "#B6B6B6",
    '7-MONTREAL': "#583697",
    '1-PITT': "#FF0000"
}

def route_style(feature):
    val = feature["properties"].get("route_long_name")
    
    # dash = "5,5" if "PROPOSED" in str(val) else None
    
    color = route_colors.get(val, "#999999")
    
    return {
        "fillColor": color,
        "color": color,
        "weight": 4,
        # "dashArray": dash,
        "fillOpacity": 0.6
    }

In [71]:
# zoning colors, add zoning
zone_colors = {
    'EMPLOYMENT LIGHT': '#ff7f0e',   # orange
    'RESIDENTIAL': '#1f77b4',        # blue
    'RURAL': '#2ca02c',              # green
    'EMPLOYMENT GENERAL': '#d62728', # red
    'INSTITUTIONAL': '#9467bd',      # purple
    'OPEN SPACE': '#8c564b',         # brown
    'COMMERCIAL': '#e377c2',         # pink
    'SPECIAL USES': '#7f7f7f',       # gray
    'FLOODPLAIN': '#17becf',         # cyan
    'NATURAL HERITAGE': '#bcbd22',   # yellow-green
}

geojson_data = datasets.get('Zoning').to_json()

# Add GeoJson layer with style function
folium.GeoJson(
    geojson_data,
    name='Zoning',
    style_function=lambda feature: {
        'fillColor': zone_colors.get(feature['properties']['Zone_Type'], 'black'),
        'color': 'black',  # boundary color
        'weight': 1,
        'fillOpacity': 0.5,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['Zone_Type', 'ZONING_LNG', 'DOCUMENT', 'Zoning'],
        aliases=['Type', 'Description', 'Document', 'Designation'],
        localize=True
    )
).add_to(m)

In [72]:
# Add buildings layer
bldg_colors = {
    'RESIDENTIAL': "#0084FF",
    'OPEN SPACE': '#8c564b',
    'SPECIAL USES': '#7f7f7f',
    'COMMERCIAL': "#DA18DA",
    'MANUFACTURING': "#D3B323",
    'INSTITUTIONAL': "#A210C7",
    'AGRICULTURAL': "#9DB65A",
    ' ': "#333333"
}
folium.GeoJson(
    datasets['Buildings_3114494096694623713'].to_json(),
    name='Buildings',
    style_function=lambda feature: {
        'fillColor': bldg_colors.get(feature['properties']['USAGE'], 'black'),
        'color': '#000000',
        'weight': 1,
        'fillOpacity': 0.4
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['USAGE', 'STORIES'],
        aliases=['Usage', 'Height'],
        localize=True
    )
).add_to(m)

In [73]:
# Add railway layer
folium.GeoJson(
    datasets['Rail'].to_json(),
    name='Railway',
    style_function=lambda feature: {
        # 'fillColor': '#000000',
        'dashArray': "5,5",
        'color': "#11B0E0",
        'weight': 1
    },
    tooltip='Railway'
).add_to(m)

In [74]:
# land use
geojson_landuse = datasets['Official_Plan_Schedule1-Landuse'].to_json()

landuse_colors = {
    'Open Space': '#8c564b',
    'Rural Area': "#FFD51B",
    'Employment': "#E69D00",
    'Institutional': "#E418D3",
    'Prime Agricultural': "#A39C34",
    'Rural Area (Special Policy)': "#7AC500",
    'Business District': "#9C007A",
    'Urban Residential': "#CCF000",
    'General Commercial': "#0026FF",
    'Environmental Constraint': "#FD3C3C",
    'Comprehensive Redevelopment Area': "#FC630B",
    'Future Study Area': "#5000AC"
}

folium.GeoJson(
    geojson_landuse,
    name='Land Use',
    style_function=lambda feature: {
        'fillColor': landuse_colors.get(feature['properties']['Land_Use'], 'black'),
        'fillOpacity': 0.4,
        'color': 'black',
        'weight': 1
    },
    show=False,
    tooltip=folium.GeoJsonTooltip(
        fields=['FID','FID_','Land_Use','LU_ABBREV'],
        aliases=['FID', 'FID', 'Designation', 'Usage Code'],
        localize=True
    )
).add_to(m)

In [75]:
# community boundaries
geojson_res_communities = datasets.get('Residential_Communities').to_json()

residential_zone_colors = {
    'CENTRETOWN': "#575757",
    'RIVERDALE': "#1074AD",
    'EAST END': "#E0C10E",
    'DOWNTOWN': "#0025F7",
    'EAMERS CORNERS': "#815B30",
    'GUINDON': "#81FF0B",
}

# Add GeoJson layer with style function
folium.GeoJson(
    geojson_res_communities,
    name='Residential Communities',
    style_function=lambda feature: {
        'fillColor': residential_zone_colors.get(feature['properties']['ZONE'], 'black'),
        'color': 'black',  # boundary color
        'weight': 1,
        'fillOpacity': 0.5,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['ZONE'],
        aliases=['Designation'],
        localize=True
    ),
    show=False
).add_to(m)

In [76]:
# Use stops as a feature group
stops_layer = folium.FeatureGroup(
    name="Bus Stops",
    overlay=True,
    control=True
)

In [77]:
# Add each day_type as a separate layer

day_layers = {}

for day_type in gdf["day_type"].unique():

    day_layers[day_type] = {}

    subset = gdf[gdf["day_type"] == day_type]

    for route in subset["route_long_name"].unique():

        route_layer = folium.FeatureGroup(name=route, show=False)

        day_layers[day_type][route] = route_layer

        route_layer.add_to(m)


for day_type in gdf["day_type"].unique():

    subset = gdf[gdf["day_type"] == day_type]

    layer = folium.FeatureGroup(name=day_type)

    for _, row in gdf.iterrows():

        layer = day_layers[row["day_type"]][row["route_long_name"]]

        popup_html = service_popup(row)

        folium.GeoJson(
            row["geometry"],
            style_function=lambda feature, val=row["route_long_name"]: route_style(
                {"properties": {"route_long_name": val}}
            ),
            tooltip=f"<b>{row['route_short_name']}</b> | {row['trip_id']}"
        ).add_child(
            folium.Popup(popup_html, max_width=500)
        ).add_to(layer)

    # for _, row in subset.iterrows():

    #     popup_html = service_popup(row)

    #     folium.GeoJson(
    #         row["geometry"],
    #         style_function=lambda feature, val=row["route_long_name"]: route_style(
    #             {"properties": {"route_long_name": val}}
    #         ),
    #         tooltip=f"<b>{row['route_short_name']}</b> | {row['trip_id']}",
    #     ).add_child(
    #         folium.Popup(popup_html, max_width=500)
    #     ).add_to(layer)

    # layer.add_to(m)
from folium.plugins import TreeLayerControl

tree = {
    "label": "Routes",
    "children": []
}

for day_type, routes in day_layers.items():

    day_node = {
        "label": day_type,
        "children": []
    }

    for route_name, layer in routes.items():

        day_node["children"].append({
            "label": route_name,
            "layer": layer
        })

    tree["children"].append(day_node)

TreeLayerControl(
    overlay_tree=tree,
    collapsed=False,
    position='topleft'
).add_to(m)

In [78]:
# Add stops with times
stops = datasets.get('stops_with_times')

min_id = stops["stop_id"].min()
max_id = stops["stop_id"].max()

def lerp(a, b, t):
    return int(a + (b - a) * t)

def stop_gradient_color(val, vmin, vmax):

    if vmax == vmin:
        return "#000000"

    t = (val - vmin) / (vmax - vmin)
    t = max(0, min(1, t))

    red = (255, 0, 0)
    black = (0, 0, 0)
    blue = (0, 0, 255)

    if t < 0.5:
        t2 = t * 2
        r = lerp(red[0], black[0], t2)
        g = lerp(red[1], black[1], t2)
        b = lerp(red[2], black[2], t2)
    else:
        t2 = (t - 0.5) * 2
        r = lerp(black[0], blue[0], t2)
        g = lerp(black[1], blue[1], t2)
        b = lerp(black[2], blue[2], t2)

    return f"#{r:02x}{g:02x}{b:02x}"

for _, row in stops.iterrows():

    import numpy as np

    popup_html = stops_popup(row)

    color = stop_gradient_color(row["stop_id"], min_id, max_id)

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color,
        fill=True,
        fill_color="white",
        fill_opacity=1,
        popup=folium.Popup(popup_html, max_width=650),
        z_index_offset=1000
    ).add_to(stops_layer)


    # folium.CircleMarker(
    #     location=[row.geometry.y, row.geometry.x],
    #     radius=5,
    #     color="black",
    #     fill=True,
    #     fill_color="white",
    #     fill_opacity=1,
    #     popup=folium.Popup(popup_html, max_width=650)
    # ).add_to(m)

stops_layer.add_to(m)

In [79]:
# Layer Ctrl
# from folium.plugins import MarkerCluster
# MarkerCluster().add_to(m)


# Add layer control
folium.LayerControl().add_to(m)

## DISPLAY

In [ ]:
m

In [ ]:
m.save('busses.html')

## DATA CHECK
Check out the data and what it looks like, then how we can play with it

In [ ]:
routes = to_pd(datasets.get('routes'))

routes.head()

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,trip_id,route_id,route_short_name,route_long_name,route_type,service_exceptions,geometry,day_type
0,MONTREAL_WeekdayT1_1:1282063:1284340,MONTREAL,7,7-MONTREAL,3,"[{'service_id': 100027, 'date': 20260305, 'exc...","LINESTRING (-74.683244 45.027914, -74.68273 45...",Weekday
1,MONTREAL_WeekdayT1_15:1279037:1284340,MONTREAL,7,7-MONTREAL,3,"[{'service_id': 100027, 'date': 20260305, 'exc...","LINESTRING (-74.728895 45.018219, -74.72871 45...",Weekday
2,MONTREAL_WeekdayT1_12:1279209:1284290,MONTREAL,7,7-MONTREAL,3,"[{'service_id': 100027, 'date': 20260305, 'exc...","LINESTRING (-74.728895 45.018219, -74.72871 45...",Weekday
3,MONTREAL_SaturdayT2_1:1279267:1284276,MONTREAL,7,7-MONTREAL,3,"[{'service_id': 100026, 'date': 20260307, 'exc...","LINESTRING (-74.728895 45.018219, -74.72871 45...",Saturday
4,MONTREAL_SaturdayT3_3:1279403:1284426,MONTREAL,7,7-MONTREAL,3,"[{'service_id': 100026, 'date': 20260307, 'exc...","LINESTRING (-74.728895 45.018219, -74.72871 45...",Saturday


In [ ]:
stops = to_pd(datasets.get('stops_with_times'))

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


In [ ]:
stops

,stop_id,stop_name,wheelchair_boarding,stop_times,geometry
0,101,Joyce & Riverdale,0,[{'trip_id': 'RIVERDALE_WeekdayT1_1:1279533:13...,POINT (-74.77124 45.016235)
1,102,Riverdale & Peter,0,[{'trip_id': 'RIVERDALE_WeekdayT1_1:1279533:13...,POINT (-74.769992 45.014459)
2,103,Riverdale & Dover,0,[{'trip_id': 'RIVERDALE_WeekdayT1_1:1279533:13...,POINT (-74.769249 45.013607)
3,104,Riverdale & Princess,0,[{'trip_id': 'RIVERDALE_WeekdayT1_1:1279533:13...,POINT (-74.768519 45.012744)
4,105,Riverdale & Queen,0,[{'trip_id': 'RIVERDALE_WeekdayT1_1:1279533:13...,POINT (-74.767618 45.011619)
...,...,...,...,...,...
303,9960,Ridgewood (Northside),0,[{'trip_id': 'BUSINESSPARK8_IndustrialT6_3:147...,POINT (-74.686156 45.038334)
304,9980,Nick Kaneb & Marleau,0,[{'trip_id': 'MCCONNELL_WeekdayT1_2:1478196:14...,POINT (-74.708092 45.036104)
305,9990,SCM Way,0,[{'trip_id': 'BUSINESSPARK9_IndustrialT4_2:148...,POINT (-74.688133 45.057864)
306,9991,Olymel,0,[{'trip_id': 'BUSINESSPARK9_IndustrialT4_2:148...,POINT (-74.685999 45.052375)


In [ ]:
zones = to_pd(datasets.get('Zoning'))
zones

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,FID,Zoning,Zone_Type,Zone_Abb,ZONING_LNG,DOCUMENT,OBJECTID,Shape__Area,Shape__Length,geometry
0,1,RES 15,RESIDENTIAL,RES,RESIDENTIAL 15,5-Residential-15.pdf,1,897.669434,148.378465,"POLYGON ((-74.700599 45.028027, -74.7006 45.02..."
1,2,RES 30,RESIDENTIAL,RES,RESIDENTIAL 30,7-Residential-30.pdf,2,6276.190430,326.546988,"POLYGON ((-74.706318 45.02629, -74.706893 45.0..."
2,3,SPU,SPECIAL USES,SPU,SPECIAL USES,29-Special-Uses.pdf,3,8735.589844,635.155740,"MULTIPOLYGON (((-74.715404 45.022046, -74.7152..."
3,4,RES 30,RESIDENTIAL,RES,RESIDENTIAL 30,7-Residential-30.pdf,4,23441.399414,693.365053,"POLYGON ((-74.712406 45.021591, -74.712461 45...."
4,5,RES 30,RESIDENTIAL,RES,RESIDENTIAL 30,7-Residential-30.pdf,5,16539.779541,543.444074,"POLYGON ((-74.73111 45.026853, -74.730372 45.0..."
...,...,...,...,...,...,...,...,...,...,...
365,383,GI-2,INSTITUTIONAL,INS,GENERAL INSTITUTIONAL,18-General-Institutional.pdf,225,105378.689453,1332.676959,"POLYGON ((-74.705513 45.040939, -74.703944 45...."
366,385,RES 20-18,RESIDENTIAL,RES,RESIDENTIAL 20,6-Residential-20.pdf,218,19803.976074,577.689806,"POLYGON ((-74.75094 45.03569, -74.750144 45.03..."
367,387,RES 20,RESIDENTIAL,RES,RESIDENTIAL 20,6-Residential-20.pdf,204,702.396484,126.046147,"POLYGON ((-74.698262 45.035445, -74.698428 45...."
368,388,RES 20,RESIDENTIAL,RES,RESIDENTIAL 20,6-Residential-20.pdf,204,38333.256836,1221.914819,"POLYGON ((-74.695872 45.035431, -74.694794 45...."


In [ ]:
list(set(zones['Zone_Type'].values))

['INSTITUTIONAL',
 'NATURAL HERITAGE',
 'SPECIAL USES',
 'RESIDENTIAL',
 'COMMERCIAL',
 'EMPLOYMENT LIGHT',
 'RURAL',
 'EMPLOYMENT GENERAL',
 'OPEN SPACE',
 'FLOODPLAIN']

In [ ]:
residential_zones = to_pd(datasets.get('Residential_Communities'))
residential_zones

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,FID,ZONE,Shape__Area,Shape__Length,geometry
0,1,EAMERS CORNERS,6.631060e+06,11040.702352,"POLYGON ((-74.777461 45.057517, -74.751908 45...."
1,2,EAST END,8.134200e+06,12909.499647,"POLYGON ((-74.725206 45.033426, -74.70772 45.0..."
2,3,DOWNTOWN,3.808168e+06,9346.117083,"POLYGON ((-74.725189 45.033397, -74.72027 45.0..."
3,4,CENTRETOWN,5.107930e+06,11093.130910,"POLYGON ((-74.716637 45.042342, -74.718798 45...."
4,5,RIVERDALE,3.163795e+06,9468.532845,"POLYGON ((-74.789705 45.032031, -74.77788 45.0..."
5,6,GUINDON,2.375273e+06,9289.999865,"POLYGON ((-74.785963 45.027216, -74.789973 45...."


In [ ]:
list(set(residential_zones['ZONE'].values))

['EAST END',
 'RIVERDALE',
 'DOWNTOWN',
 'GUINDON',
 'EAMERS CORNERS',
 'CENTRETOWN']

In [ ]:
bldgs = to_pd(datasets['Buildings_3114494096694623713'])
bldgs.head()

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,FID,STORIES,USAGE,geometry
0,1,1,RESIDENTIAL,"POLYGON ((-74.780112 45.017127, -74.780078 45...."
1,2,1,RESIDENTIAL,"POLYGON ((-74.780762 45.015855, -74.780712 45...."
2,3,1,RESIDENTIAL,"POLYGON ((-74.774252 45.01422, -74.774307 45.0..."
3,4,1,RESIDENTIAL,"POLYGON ((-74.771442 45.034141, -74.771408 45...."
4,5,1,OPEN SPACE,"POLYGON ((-74.834082 45.032359, -74.834144 45...."


In [ ]:
list(set(bldgs['USAGE']))

[' ',
 'INSTITUTIONAL',
 'SPECIAL USES',
 'RESIDENTIAL',
 'COMMERCIAL',
 'AGRICULTURAL',
 'MANUFACTURING',
 'OPEN SPACE']

In [ ]:
rail = to_pd(datasets.get('Rail'))
rail

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,FID,LAYER,geometry
0,1,RAILWAY,"LINESTRING (-74.704397 45.041565, -74.703899 4..."
1,2,RAILWAY,"LINESTRING (-74.704397 45.041565, -74.704268 4..."
2,3,RAILWAY,"LINESTRING (-74.772516 45.042218, -74.772791 4..."
3,4,RAILWAY,"LINESTRING (-74.77291 45.042255, -74.773074 45..."
4,5,RAILWAY,"LINESTRING (-74.772516 45.042218, -74.772815 4..."
5,6,RAILWAY,"LINESTRING (-74.772996 45.042216, -74.773064 4..."
6,7,RAILWAY,"LINESTRING (-74.773074 45.042251, -74.77341 45..."
7,8,RAILWAY,"LINESTRING (-74.774812 45.041667, -74.774776 4..."
8,9,RAILWAY,"LINESTRING (-74.805054 45.04221, -74.805248 45..."
9,10,RAILWAY,"LINESTRING (-74.674839 45.053676, -74.679735 4..."


In [ ]:
OP1 = to_pd(datasets['Official_Plan_Schedule1-Landuse'])
OP1

/home/inpw/Application/corndv/sources.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


,FID,FID_,Land_Use,LU_ABBREV,Shape__Area,Shape__Length,geometry
0,1,0,Rural Area,RA,672556.217041,4191.951801,"POLYGON ((-74.834393 45.034504, -74.834139 45...."
1,2,0,Rural Area,RA,41016.097900,1683.016830,"POLYGON ((-74.813684 45.042404, -74.813438 45...."
2,3,0,Rural Area,RA,85700.831299,1476.515123,"POLYGON ((-74.807285 45.044987, -74.806408 45...."
3,4,0,Rural Area,RA,925705.665771,6934.067130,"POLYGON ((-74.7698 45.045369, -74.767891 45.04..."
4,5,0,Rural Area,RA,174977.392822,2075.919104,"POLYGON ((-74.787735 45.053091, -74.786899 45...."
...,...,...,...,...,...,...,...
99,100,0,General Commercial,GC,10108.088867,432.914656,"POLYGON ((-74.729178 45.033654, -74.729344 45...."
100,101,0,Comprehensive Redevelopment Area,CRA,19402.379150,751.885549,"POLYGON ((-74.7297 45.03165, -74.731021 45.031..."
101,102,0,Urban Residential,U RES,10.715088,651.831850,"POLYGON ((-74.760506 45.02089, -74.762512 45.0..."
102,103,0,Open Space,O SP,53883.963379,2063.588770,"POLYGON ((-74.7614 45.01706, -74.761693 45.016..."


In [ ]:
list(set(OP1['Land_Use'].values))

['Open Space',
 'Rural Area',
 'Employment',
 'Institutional',
 'Prime Agricultural',
 'Rural Area (Special Policy)',
 'Business District',
 'Urban Residential',
 'General Commercial',
 'Environmental Constraint',
 'Comprehensive Redevelopment Area',
 'Future Study Area']